In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch

print("إصدار PyTorch:", torch.version)
print("GPU متاح:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("نوع GPU:", torch.cuda.get_device_name(0))

إصدار PyTorch: <module 'torch.version' from '/usr/local/lib/python3.13/dist-packages/torch/version.py'>
GPU متاح: True
نوع GPU: Tesla T4


In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(repo_id="gasparyanartur/things-eeg2", repo_type="dataset")
print("أول 30 ملف في المستودع:")
for f in files[:30]:
    print(f)

print("\nMetadata files :")
for f in files:
    if any(k in f.lower() for k in ['image', 'meta', 'stim', 'clip', 'test']):
        if 'sub-02' not in f and 'sub-03' not in f: # نستثني باقي الأشخاص
            print(f)

أول 30 ملف في المستودع:
.gitattributes
LICENSE.txt
README.md
image_metadata.npy
imgs/test_images.zip
imgs/training_images.zip
raw-eeg/sub-01/ses-01/raw_eeg_test.npy
raw-eeg/sub-01/ses-01/raw_eeg_training.npy
raw-eeg/sub-01/ses-02/raw_eeg_test.npy
raw-eeg/sub-01/ses-02/raw_eeg_training.npy
raw-eeg/sub-01/ses-03/raw_eeg_test.npy
raw-eeg/sub-01/ses-03/raw_eeg_training.npy
raw-eeg/sub-01/ses-04/raw_eeg_test.npy
raw-eeg/sub-01/ses-04/raw_eeg_training.npy
raw-eeg/sub-02/ses-01/raw_eeg_test.npy
raw-eeg/sub-02/ses-01/raw_eeg_training.npy
raw-eeg/sub-02/ses-02/raw_eeg_test.npy
raw-eeg/sub-02/ses-02/raw_eeg_training.npy
raw-eeg/sub-02/ses-03/raw_eeg_test.npy
raw-eeg/sub-02/ses-03/raw_eeg_training.npy
raw-eeg/sub-02/ses-04/raw_eeg_test.npy
raw-eeg/sub-02/ses-04/raw_eeg_training.npy
raw-eeg/sub-03/ses-01/raw_eeg_test.npy
raw-eeg/sub-03/ses-01/raw_eeg_training.npy
raw-eeg/sub-03/ses-02/raw_eeg_test.npy
raw-eeg/sub-03/ses-02/raw_eeg_training.npy
raw-eeg/sub-03/ses-03/raw_eeg_test.npy
raw-eeg/sub-03/

In [ ]:
import os
import zipfile
import numpy as np
from huggingface_hub import hf_hub_download

# 1. إنشاء مجلد المشروع على Google Drive تلقائياً
drive_dir = "/content/drive/MyDrive/Brain_to_Image"
os.makedirs(drive_dir, exist_ok=True)
print(f"مجلد الحفظ على درايف جاهز: {drive_dir}")

# 2. تحميل ملف الـ Metadata وملفات الصور المضغوطة
print("\nجاري تحميل image_metadata.npy وملفات الصور...")
meta_path = hf_hub_download(repo_id="gasparyanartur/things-eeg2", filename="image_metadata.npy", repo_type="dataset")
train_zip = hf_hub_download(repo_id="gasparyanartur/things-eeg2", filename="imgs/training_images.zip", repo_type="dataset")
test_zip = hf_hub_download(repo_id="gasparyanartur/things-eeg2", filename="imgs/test_images.zip", repo_type="dataset")

# 3. فك ضغط الصور على مساحة التخزين في كولاب (Disk)
extract_dir = "/content/things_images"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(train_zip, 'r') as z:
    z.extractall(extract_dir)
with zipfile.ZipFile(test_zip, 'r') as z:
    z.extractall(extract_dir)
print("تم فك ضغط الصور بنجاح!")

# 4. استكشاف محتوى image_metadata.npy ومجلدات الصور
meta = np.load(meta_path, allow_pickle=True).item()
print("\n=== مفاتيح ملف image_metadata.npy ===")
for k, v in meta.items():
    print(f"- {k}: الطول = {len(v)} | أول عنصر: {v[0]}")

print("\n=== المجلدات الناتجة بعد فك الضغط ===")
print(os.listdir(extract_dir))

مجلد الحفظ على درايف جاهز: /content/drive/MyDrive/Brain_to_Image

جاري تحميل image_metadata.npy وملفات الصور...


image_metadata.npy: reconstructing file:   0%|          |  0.00B /  657kB            

image_metadata.npy: downloading bytes:           |  0.00B            

imgs/training_images.zip: reconstructing file:   0%|          |  0.00B /  655MB            

imgs/training_images.zip: downloading bytes:           |  0.00B            

imgs/test_images.zip: reconstructing file:   0%|          |  0.00B / 8.13MB            

imgs/test_images.zip: downloading bytes:           |  0.00B            

تم فك ضغط الصور بنجاح!

=== مفاتيح ملف image_metadata.npy ===
- test_img_concepts: الطول = 200 | أول عنصر: 00001_aircraft_carrier
- test_img_concepts_THINGS: الطول = 200 | أول عنصر: 00010_aircraft_carrier
- test_img_files: الطول = 200 | أول عنصر: aircraft_carrier_06s.jpg
- train_img_files: الطول = 16540 | أول عنصر: aardvark_01b.jpg
- train_img_concepts: الطول = 16540 | أول عنصر: 00001_aardvark
- train_img_concepts_THINGS: الطول = 16540 | أول عنصر: 00001_aardvark

=== المجلدات الناتجة بعد فك الضغط ===
['test_images', 'training_images']


In [ ]:
import os
import gc
import zipfile
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive
from huggingface_hub import hf_hub_download
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection

# 1. إعادة ربط Google Drive بعد الكراش
drive.mount('/content/drive')
drive_dir = "/content/drive/MyDrive/Brain_to_Image"
os.makedirs(drive_dir, exist_ok=True)

# 2. تحميل الـ Metadata والتأكد من وجود الصور على القرص
meta_path = hf_hub_download(repo_id="gasparyanartur/things-eeg2", filename="image_metadata.npy", repo_type="dataset")
meta = np.load(meta_path, allow_pickle=True).item()

extract_dir = "/content/things_images"
if not os.path.exists(os.path.join(extract_dir, "training_images")):
    print("جاري تحميل وفك ضغط الصور...")
    train_zip = hf_hub_download(repo_id="gasparyanartur/things-eeg2", filename="imgs/training_images.zip", repo_type="dataset")
    test_zip = hf_hub_download(repo_id="gasparyanartur/things-eeg2", filename="imgs/test_images.zip", repo_type="dataset")
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(train_zip, 'r') as z:
        z.extractall(extract_dir)
    with zipfile.ZipFile(test_zip, 'r') as z:
        z.extractall(extract_dir)
else:
    print("الصور موجودة مسبقاً على القرص، تم تخطي التحميل!")

# 3. تجهيز مسارات الصور
sample_path_nested = os.path.join(extract_dir, "training_images", meta['train_img_concepts'][0], meta['train_img_files'][0])
if os.path.exists(sample_path_nested):
    train_paths = [os.path.join(extract_dir, "training_images", c, f) for c, f in zip(meta['train_img_concepts'], meta['train_img_files'])]
    test_paths = [os.path.join(extract_dir, "test_images", c, f) for c, f in zip(meta['test_img_concepts'], meta['test_img_files'])]
else:
    train_paths = [os.path.join(extract_dir, "training_images", f) for f in meta['train_img_files']]
    test_paths = [os.path.join(extract_dir, "test_images", f) for f in meta['test_img_files']]

# 4. تحميل قسم الرؤية فقط من CLIP بصيغة float16 لتوفير الرام وتسريع الـ GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-large-patch14"
print(f"\nجاري تحميل CLIP Vision فقط على ({device})...")

processor = CLIPImageProcessor.from_pretrained(model_id)
vision_model = CLIPVisionModelWithProjection.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
vision_model.eval()

# 5. استخراج المتجهات مع إدارة صارمة للرام
def extract_clip_embeddings(image_paths, batch_size=64):
    all_embeddings = []
    with torch.no_grad():
        for step, i in enumerate(tqdm(range(0, len(image_paths), batch_size))):
            batch_paths = image_paths[i:i + batch_size]
            images = []
            for p in batch_paths:
                with Image.open(p) as img:
                    images.append(img.convert("RGB"))

            inputs = processor(images=images, return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device, dtype=vision_model.dtype)

            outputs = vision_model(pixel_values=pixel_values).image_embeds
            outputs = torch.nn.functional.normalize(outputs.float(), dim=-1)
            all_embeddings.append(outputs.cpu())

            # تنظيف فوري لذاكرة الدفعة
            del images, inputs, pixel_values, outputs
            if step % 25 == 0:
                gc.collect()

    return torch.cat(all_embeddings, dim=0)

print("\nجاري استخراج متجهات صور التدريب (16,540 صورة)...")
clip_train_embeddings = extract_clip_embeddings(train_paths, batch_size=64)

print("\nجاري استخراج متجهات صور الاختبار (200 صورة)...")
clip_test_embeddings = extract_clip_embeddings(test_paths, batch_size=64)

# 6. الحفظ الدائم على Google Drive
torch.save(clip_train_embeddings, os.path.join(drive_dir, "clip_train_embeddings.pt"))
torch.save(clip_test_embeddings, os.path.join(drive_dir, "clip_test_embeddings.pt"))
np.save(os.path.join(drive_dir, "image_metadata.npy"), meta)

print("\n=== تمت المرحلة الأولى بنجاح! ===")
print(f"أبعاد متجهات التدريب: {clip_train_embeddings.shape}")
print(f"أبعاد متجهات الاختبار: {clip_test_embeddings.shape}")
print(f"تم الحفظ في: {drive_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


الصور موجودة مسبقاً على القرص، تم تخطي التحميل!

جاري تحميل CLIP Vision فقط على (cuda)...


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{


جاري استخراج متجهات صور التدريب (16,540 صورة)...


  0%|          | 0/259 [00:00<?, ?it/s]


جاري استخراج متجهات صور الاختبار (200 صورة)...


  0%|          | 0/4 [00:00<?, ?it/s]


=== تمت المرحلة الأولى بنجاح! ===
أبعاد متجهات التدريب: torch.Size([16540, 768])
أبعاد متجهات الاختبار: torch.Size([200, 768])
تم الحفظ في: /content/drive/MyDrive/Brain_to_Image
